In [0]:
%sql
use catalog projectcatalog;
--use schema sparkschemaprojectsales

In [0]:
%sql
create schema SliverSchemaSales
    


In [0]:
%sql
use SliverSchemaSales

In [0]:
%sql
select * from sparkschemaprojectsales.tblprjcustomersource

customer_id,first_name,last_name,email,city,state
C01001,Darrell,Ward,waynedunlap@yahoo.com,Lake Tamaraberg,MA
C01002,Anthony,Phillips,mariajohnson@mccullough.biz,Danielhaven,WA
C01003,Jonathan,Ortega,conniejohnson@hotmail.com,Lisashire,IN
C01004,Joseph,Simon,dawnmendoza@stanton.net,Villashire,VA
C01005,Donna,Reyes,ismith@wright-haas.com,Lake Summerstad,IL
C01006,Robert,Patel,rodney54@brown.org,East Michaelshire,WV
C01007,James,King,gdixon@yahoo.com,Warnerville,ME
C01008,Edward,Allen,pochoa@pollard.com,South Brendaberg,NH
C01009,Bryan,Rivas,campbellcynthia@wilson.com,South James,RI
C01010,Thomas,Wilson,johnpadilla@hotmail.com,Austinburgh,AZ


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vwPrjCustomertrimmed as
select trim(customer_id) as customer_id,trim(first_name) as first_name,trim(last_name) as last_name,trim(email) as email,trim(city) as city,trim(state) as state
from sparkschemaprojectsales.tblprjcustomersource

In [0]:
%sql
create or replace temp view vwprojcustomersdistinct as
select distinct customer_id,first_name,last_name,email,city,state
from vwPrjCustomertrimmed
    


In [0]:
%sql
select customer_id,first_name,last_name,email,city,state
from vwprojcustomersdistinct
where customer_id is null or first_name is null or last_name is null or email is null or city is null or state is null

customer_id,first_name,last_name,email,city,state


In [0]:
%sql
create or replace temp view vwtblprjcustomerblank as
select 
nullif(customer_id,'') as customer_id,
nullif(first_name,'') as first_name,
nullif(last_name,'') as last_name,
nullif(email,'') as email,
nullif(city,'') as city,
nullif(state,'') as state
 from vwprojcustomersdistinct


In [0]:
%sql
create or replace temp view vwprjcustomerstandardized as
select 
customer_id,
initcap(lower(first_name)) as first_name,
initcap(lower(last_name)) as last_name,
lower(email) as email,
initcap(lower(city)) as city,
upper(state) as state
from vwtblprjcustomerblank

In [0]:
%sql
select * from vwprjcustomerstandardized where email is null or email not rlike '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'

customer_id,first_name,last_name,email,city,state


In [0]:
%sql
create or replace temp view vwprjcustomeremail as
select * from vwprjcustomerstandardized where email rlike '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'


In [0]:
%sql
select email,count(*) as cnt from vwprjcustomeremail
group by email
having count(*)  > 1
order by cnt desc


email,cnt
kenneth33@hotmail.com,2


In [0]:
%sql
create or replace temp view vwprjcustomeremailunique as
select * from 
(select *, row_number() over(partition by email order by customer_id ) as rn from vwprjcustomeremail)
duplicatecheck 
where rn = 1

In [0]:
%sql
select * from vwprjcustomeremailunique where customer_id is null or customer_id 
not rlike '^C[0-9]{5}$'

customer_id,first_name,last_name,email,city,state,rn


In [0]:
%sql
create or replace temp view vwprjcustomervalidid as
select * from vwprjcustomeremailunique where customer_id rlike '^C[0-9]{5}$'


In [0]:
%sql
select * from vwprjcustomervalidid where state is null or state not rlike '^[A-Z]{2}$'

    


customer_id,first_name,last_name,email,city,state,rn


In [0]:
%sql
create or replace temp view vwprjcustomerstate as
select * from vwprjcustomervalidid where state rlike '^[A-Z]{2}$'

In [0]:
%sql
create or replace temp view vwprjcustomeraddcoloumn as
select customer_id,first_name,last_name,
concat_ws(' ', first_name, last_name) as fullname,
email,city,state
from vwprjcustomerstate
    


In [0]:
%sql
create or replace temp view vwprjcustomerfinal as
select customer_id,first_name,last_name, fullname,email,city,state,
current_timestamp() as created_timesatmp,
'customersfile.parquet' as sourcefile
from vwprjcustomeraddcoloumn


In [0]:
%sql
create or replace table tblPrjCustomerSilver 
using delta
as
select * from vwprjcustomerfinal

num_affected_rows,num_inserted_rows


In [0]:
%sql
select count(*) as bronzedatacnt from sparkschemaprojectsales.tblprjcustomersource


bronzedatacnt
2000


In [0]:
%sql
select count(*) as silverdatacnt from tblPrjCustomerSilver

silverdatacnt
1999


In [0]:
%sql
select * from tblprjcustomersilver

customer_id,first_name,last_name,fullname,email,city,state,created_timesatmp,sourcefile
C01965,Chad,Martinez,Chad Martinez,aaron30@gmail.com,Rojasview,FL,2026-08-11T10:23:27.588Z,customersfile.parquet
C00410,Justin,Henry,Justin Henry,aaron67@yahoo.com,Andrewburgh,WV,2026-08-11T10:23:27.588Z,customersfile.parquet
C01903,Beth,Smith,Beth Smith,aaron98@gmail.com,Gallegosport,AK,2026-08-11T10:23:27.588Z,customersfile.parquet
C01683,Stephanie,Peterson,Stephanie Peterson,aarongriffith@moore.com,Ericfort,SC,2026-08-11T10:23:27.588Z,customersfile.parquet
C01490,Christopher,Guerra,Christopher Guerra,aaronjackson@hill-smith.com,East Kaylamouth,VA,2026-08-11T10:23:27.588Z,customersfile.parquet
C00119,Katie,Davis,Katie Davis,aaronroberts@hotmail.com,Lake Robertland,KS,2026-08-11T10:23:27.588Z,customersfile.parquet
C01804,Hannah,Rogers,Hannah Rogers,abarajas@hotmail.com,New Davidview,KY,2026-08-11T10:23:27.588Z,customersfile.parquet
C00451,Evelyn,Wolfe,Evelyn Wolfe,abigail73@jones.info,Craigfurt,WV,2026-08-11T10:23:27.588Z,customersfile.parquet
C01065,Alexander,Lara,Alexander Lara,abigailbanks@hotmail.com,South Amy,IL,2026-08-11T10:23:27.588Z,customersfile.parquet
C00198,Autumn,Williams,Autumn Williams,abigailray@perez.com,New Josephhaven,WA,2026-08-11T10:23:27.588Z,customersfile.parquet


In [0]:
bronze_df = spark.read.table("tblprjcustomersource")

In [0]:
display(bronze_df)

customer_id,first_name,last_name,email,city,state
C01001,Darrell,Ward,waynedunlap@yahoo.com,Lake Tamaraberg,MA
C01002,Anthony,Phillips,mariajohnson@mccullough.biz,Danielhaven,WA
C01003,Jonathan,Ortega,conniejohnson@hotmail.com,Lisashire,IN
C01004,Joseph,Simon,dawnmendoza@stanton.net,Villashire,VA
C01005,Donna,Reyes,ismith@wright-haas.com,Lake Summerstad,IL
C01006,Robert,Patel,rodney54@brown.org,East Michaelshire,WV
C01007,James,King,gdixon@yahoo.com,Warnerville,ME
C01008,Edward,Allen,pochoa@pollard.com,South Brendaberg,NH
C01009,Bryan,Rivas,campbellcynthia@wilson.com,South James,RI
C01010,Thomas,Wilson,johnpadilla@hotmail.com,Austinburgh,AZ


In [0]:
%sql
describe tblPrjCustomerSource

col_name,data_type,comment
customer_id,string,null
first_name,string,null
last_name,string,null
email,string,null
city,string,null
state,string,null


In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("tblPrjCustomerSource")

silver_df = (
    bronze_df
    # Remove duplicate customers
    .dropDuplicates(["customer_id"])
    # Trim spaces
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("email", trim(col("email")))
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    # checking null values
    .filter(col("customer_id").isNotNull())
    .filter(col("email").isNotNull())
    .filter(col("first_name").isNotNull())
    .filter(col("last_name").isNotNull())
    .filter(col("city").isNotNull())
    .filter(col("state").isNotNull())
    # replacing blank values with null
    

    # Email validation
    .filter(col("email").contains("@"))

    # Standardization
    .withColumn("customer_id", upper(col("customer_id")))
    .withColumn("email", lower(col("email")))
    .withColumn("first_name", initcap(col("first_name")))
    .withColumn("last_name", initcap(col("last_name")))
    .withColumn("city", initcap(col("city")))
    .withColumn("state", upper(col("state")))

    # Remove empty strings
    .filter(col("customer_id") != "")
    .filter(col("email") != "")
)

display(silver_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8938870415533448>, line 42
      3 bronze_df = spark.table("tblPrjCustomerSource")
      5 silver_df = (
      6     bronze_df
      7     # Remove duplicate customers
   (...)
     39     .filter(col("email") != "")
     40 )
---> 42 display(silver_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an e